# 06 — Evaluation Expansion (Week 8)

Metrics beyond R², on the same homes for every finalist:

- **MdAPE** (headline), **MAPE**, **R²**, **MAE**, **RMSE**, the share of homes within 5 / 10 / 20 %, and median signed error (bias)
- Error by **price band** (sale-price quintiles, plus quintiles of the model's own estimate), **county**, **living area** and **home age**
- A **3-month backtest**, and a **paired bootstrap** that tests whether MdAPE gaps are bigger than noise

Deliverable: `data/processed/metrics_summary.csv`. It's written by `scripts/run_evaluation.py`, which this notebook mirrors. It stays git-ignored, like all outputs.

In [ ]:
import sys, json
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from idx_avm import config
from idx_avm.pipeline import prepare_dataset
from idx_avm.features import FEATURE_SETS
from idx_avm.artifacts import load_tuned_params
from idx_avm.models import fit_and_score
from idx_avm.evaluation import full_metrics, price_band_edges, band_labels, segment_breakdowns, paired_bootstrap

selected = json.loads((config.PROCESSED_DIR / "best_model_metrics.json").read_text())
PARAMS = load_tuned_params()
X_MONTHS = int(selected["n_train_months"])
data = prepare_dataset(week6_features=True)
DF, TEST_MONTH = data.df, data.months[-1]
FULL = list(FEATURE_SETS)[-1]
MODELS = {
    "Linear Regression (baseline)": ("Linear Regression", "Base (Week 4)"),
    "Random Forest": ("Random Forest", "Base (Week 4)"),
    f"{selected['selected_model']} (selected)": (selected["selected_model"], selected["selected_feature_set"]),
    "Gradient Boosting + all features": ("Gradient Boosting", FULL),
}
# If the selection is one of the fixed comparison models, keep only its "(selected)" entry
SEL = (selected["selected_model"], selected["selected_feature_set"])
MODELS = {k: v for k, v in MODELS.items() if k.endswith("(selected)") or v != SEL}
list(MODELS)

## 1. Score every model on the same homes, for 3 months

In [ ]:
backtest, preds = [], {}
for label, (model, fs) in MODELS.items():
    for k in (2, 1, 0):
        r = fit_and_score(DF, TEST_MONTH - k, X_MONTHS, model, PARAMS[model], FEATURE_SETS[fs], fs)
        backtest.append({"label": label, "test_month": str(TEST_MONTH - k), **full_metrics(r.test[config.TARGET], r.test["Predicted"])})
        if k == 0:
            preds[label] = r.test
backtest = pd.DataFrame(backtest)
backtest.pivot(index="test_month", columns="label", values="MdAPE").round(2)

## 2. metrics_summary

In [ ]:
summary = pd.DataFrame([{"label": lab, **full_metrics(t[config.TARGET], t["Predicted"])} for lab, t in preds.items()])
summary.set_index("label")[["MdAPE", "MAPE", "R2", "MAE", "RMSE", "within_5pct", "within_10pct", "within_20pct", "median_signed_error_pct"]].round(3)

## 3. Which price bands perform better?

In [ ]:
base_label = list(MODELS)[0]
y = preds[base_label][config.TARGET]
edges = price_band_edges(y)
labels = band_labels(y, edges)
seg = pd.concat([segment_breakdowns(t, edges, labels).assign(label=lab) for lab, t in preds.items()], ignore_index=True)
band = seg[seg.segment_type == "Price band (sale price)"].pivot(index="segment", columns="label", values="MdAPE").loc[labels]
band.plot.bar(figsize=(10, 4), ylabel="MdAPE (%)", rot=0); plt.title("MdAPE by sale-price quintile"); plt.show()
band.round(2)

In [ ]:
sel_label = [l for l in MODELS if l.endswith("(selected)")][0]
bias = seg[(seg.label == sel_label) & seg.segment_type.str.startswith("Price band")].pivot(
    index="segment", columns="segment_type", values="median_signed_error_pct").loc[labels]
bias.plot.bar(figsize=(10, 4), ylabel="Median signed error (%)", rot=0); plt.axhline(0, color="gray"); plt.show()
bias.round(2)

## 4. Geography, size, age

In [ ]:
for kind in ("County", "Living area", "Home age"):
    display(seg[seg.segment_type == kind].pivot_table(index="segment", columns="label", values="MdAPE", sort=False).round(2))

## 5. Is the improvement bigger than noise? (paired bootstrap)

In [ ]:
sel = preds[sel_label]
pd.DataFrame([{"vs": lab, **paired_bootstrap(sel[config.TARGET], sel["Predicted"], t["Predicted"])}
              for lab, t in preds.items() if lab != sel_label]).round(3)

## 6. Insights (fill in from the run)

`scripts/run_evaluation.py` writes data-driven versions of these to `eval_insights.json`, and the dashboard's Evaluation page shows them.

- Which price bands are most and least accurate, and where the gain over the baseline is largest.
- Over- and under-estimation by band, comparing grouping by sale price with grouping by the model's estimate.
- Counties with the highest error.
- Whether the gain is stable across the 3 months and outside the bootstrap noise band.